# 项目总览：从可靠验证到 Round 2 特征工程

这个 Notebook 是整个项目的**思路地图与阶段复盘**。训练代码集中在 `src/`，逐折证据保存在 `results/`，各专题 Notebook 负责展开细节；这里重点说明：为什么实验顺序会这样演变、每一步解决了什么问题，以及当前证据把下一步指向哪里。

当前主线已经从“建立一个能工作的模型”推进到“测量新增特征在 FE10 之上的真实增量”：

```text
理解数据
→ 跑通 Baseline
→ 建立可信 5-Fold CV
→ 校准 CatBoost 训练预算
→ Round 1 找到 FE10
→ 用 NULL 校准 feature-addition noise
→ Round 2 按 family 跑完整 5-Fold
→ 只对赢家做 LOO / Reserve 扩展
→ 第二个 CV seed 复核
→ 最终 Kaggle 验证
```

---

## Table of Contents

1. [当前状态与核心结论](#overview-current-state)
2. [数据理解与第一条 Baseline](#overview-data-baseline)
3. [先把验证和训练预算做可靠](#overview-validation)
4. [Round 1：从 BASE 收敛到 FE10](#overview-round1)
5. [为什么 Round 2 先做 NULL](#overview-null)
6. [Round 2 第一阶段：按 family 做完整五折](#overview-round2)
7. [当前决策与后续路线](#overview-next)
8. [文件分工与关键数字](#overview-files)


<a id="overview-current-state" name="overview-current-state"></a>

## 1. 当前状态与核心结论

当前正式 Feature Reference 是 **FE10 — `unknown_activity_time`**：

```python
unknown_activity_time = (
    daily_screen_time_hours
    - social_media_hours
    - gaming_hours
    - work_study_hours
)
```

| 当前证据 | 结果 | 含义 |
|---|---:|---|
| 无特征工程 CatBoost | `0.964007 ± 0.000473` | Round 1 的原始 BASE |
| FE10 5-Fold | **`0.964629 ± 0.000523`** | 当前正式 Feature Reference |
| FE10 vs BASE | **`+0.000621`，5/5 Fold 为正** | 已确认的稳定增量 |
| FE10 Public LB | **`0.96606`** | 本地提升得到线上旁证 |
| NULL 校准 | 15/15 fits 完成 | 已测得“只增加一列”造成的漂移 |
| Round 2 第一阶段 | 6 组 × 5-Fold 完成 | 已从 27 个候选收敛到 family 级判断 |

第一阶段最重要的最新发现是：

- `R_GROUP` 是唯一在 5/5 Fold 上都高于 FE10 的数值方向，平均增量 `+0.000079`，值得进入 LOO。
- `O_GROUP` 为弱正向（`+0.000048`，4/5 Fold），但尚未明显区别于 NULL 漂移，不能直接宣布 ratio 被“平反”。
- FE26 基本为零；FE27 5/5 Fold 为负，screen mean 方向关闭。
- `CAT_NUM_GROUP` 明显为负；`CAT2_GROUP` 也没有正向增量，对应 Reserve 不自动开启。

因此项目已经不再处于“继续想更多特征”的阶段，而是进入：**拆解 R_GROUP，确认真正贡献成员，并控制后续扩展范围。**

In [ ]:
import platform
import sys

import catboost
import ipykernel
import jupyterlab
import matplotlib
import numpy
import pandas
import sklearn

print("Python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("catboost:", catboost.__version__)
print("matplotlib:", matplotlib.__version__)
print("ipykernel:", ipykernel.__version__)
print("jupyterlab:", jupyterlab.__version__)

<a id="overview-data-baseline" name="overview-data-baseline"></a>

## 2. 数据理解与第一条 Baseline

对应 Notebook：[01_eda.ipynb](01_eda.ipynb)、[02_baseline.ipynb](02_baseline.ipynb)

| 项目 | 结论 | 对后续实验的影响 |
|---|---|---|
| Train / Test | 691,369 / 296,302 行 | 数据规模足以支持稳定五折，但单次训练成本较高 |
| Target | `addicted_label`，二分类 | 提交正类概率，评价使用 AUC |
| 原始特征 | 9 个数值 + 3 个类别 | CatBoost 可低预处理成本地建模 |
| 类别比例 | 正类约 70.9% | CV 使用 `StratifiedKFold` 保持比例 |
| 缺失 | 所有预测特征都有不同程度缺失 | 数值保留 `NaN`，类别缺失填为 `Missing` |
| `id` | 基本为唯一标识 | 从训练特征中移除 |

第一版 CatBoost 使用 500 轮和单次 80/20 Holdout：

- Holdout AUC：`0.950917`
- Public LB：`0.95227`
- 最佳轮数：`500 / 500`

本地与线上分数接近，说明流程没有明显脱节；但最佳轮数撞上限说明模型尚未训练充分。这个 Baseline 最重要的作用不是给出高分，而是同时暴露了两个问题：**单次划分不够稳，500 轮也不是收敛点。**

<a id="overview-validation" name="overview-validation"></a>

## 3. 先把验证和训练预算做可靠

对应 Notebook：[03_cv_experiments.ipynb](03_cv_experiments.ipynb)

我先用 5-Fold 验证 Holdout 是否具有代表性，再在不改变特征的情况下逐步增加训练预算。

| 阶段 | 设置 | CV / Fold 结果 | 学到什么 |
|---|---|---:|---|
| Baseline CV | 500 轮 | `0.951449 ± 0.000527` | Holdout 没有明显失真，但模型训练不足 |
| Exp 01 | 上限 3,000 + Early Stopping | `0.962744 ± 0.000459` | 训练预算是主要瓶颈，且仍接近上限 |
| 单折 Probe | 上限 10,000 | Fold 1 `0.963351`，best iter 8,044 | 找到大致收敛区间 |
| 正式无 FE 模型 | 上限 10,000，ESR 200 | **`0.964007 ± 0.000473`** | 五折 best iter 均值 8,223 |

全量训练固定 8,223 轮后，Public LB 达到 `0.96538`。这一阶段确立了两条后续纪律：

1. Kaggle LB 只用于阶段性外部验证，本地筛选依靠固定 5-Fold。
2. 在改变特征前，先固定模型、Fold、seed、线程数和 early stopping；否则差异无法归因。

### 承前启后的小总结

到这里，项目已经把“验证不稳定”和“训练不足”两个混杂因素拆开解决。后续新增特征如果带来提升，就更有理由解释为特征本身的增量，而不是训练预算或划分变化。

<a id="overview-round1" name="overview-round1"></a>

## 4. Round 1：从 BASE 收敛到 FE10

对应 Notebook：[04_feature_engineering.ipynb](04_feature_engineering.ipynb)

Round 1 测试了缺失模式、时间差与比例、活动时间构成、行为频率、睡眠与离线结构。最清晰的信号来自活动时间预算：

| 实验 | 新增表达 | 平均 AUC | 平均 Δ vs BASE | 正向 Fold |
|---|---|---:|---:|---:|
| FE09 | 已知活动时间之和 | `0.964348` | `+0.000341` | 5/5 |
| FE10 | 未解释屏幕时间 | **`0.964629`** | **`+0.000621`** | **5/5** |
| FE19 | FE09 + FE10 | `0.964651` | `+0.000643` | 5/5 vs BASE |

FE19 的平均 AUC 比 FE10 只高约 `0.000022`，直接逐折比较时仅 2/5 Fold 胜过 FE10。它增加了复杂度，却没有提供方向一致的增量，因此最终保留更简单的 FE10。

FE10 随后的 Public LB 为 `0.96606`，相对无 FE 模型的 `0.96538` 提升约 `+0.00068`，与本地 CV 的提升方向一致。

Round 1 的结论不是“所有比例都无效”，而是：**FE10 已经吸收了最清晰的时间构成信号，后续必须研究 candidate 在 FE10 之上的 incremental value。**

<a id="overview-null" name="overview-null"></a>

## 5. 为什么 Round 2 先做 NULL

对应 Notebook：[05_feature_engineering_2.ipynb](05_feature_engineering_2.ipynb)

Round 1 使用的 `0.000473` 是 BASE 五个 Fold 的**绝对 AUC 波动**。Round 2 真正关心的是同一个 Fold 下：

```text
delta_vs_reference
= AUC(FE10 + candidate) - AUC(FE10)
```

同折相减会抵消“某个 Fold 本身更容易或更困难”的影响，所以不能继续用绝对 AUC Std 机械判断增量。更贴近实际的问题是：**只多加一列、但不增加真实预测信息时，CatBoost 的 paired AUC 会漂多少？**

NULL 校准共完成 15 fits：

| 对照 | Mean Δ | Mean |Δ| | Max |Δ| | 正向 Fold |
|---|---:|---:|---:|---:|
| NULL-A：严格 affine 冗余 | `+0.000016` | `0.000051` | `0.000148` | 3/5 |
| NULL-B：permutation 1001 | `-0.000098` | `0.000121` | `0.000176` | 1/5 |
| NULL-B：permutation 1002 | `-0.000122` | `0.000122` | `0.000182` | 0/5 |

这组结果说明：

- 严格冗余列的五折均值接近零，但单 Fold 仍可能漂到约 `±0.00015`。
- 随机 permutation 列在两个 seed 上都表现出轻微负偏移，因此 NULL-A 与 NULL-B 不宜被粗暴合并成一个正式 p-value。
- 历史 FE07 `+0.000081`、FE16 `+0.000108` 的单 Fold 正信号落在 NULL 单折漂移范围内，不能单凭旧 Fold 1 结果证明有效。

因此 Round 2 放弃“27 个候选全部 Fold 1 排名”，改用 **family-level 完整 5-Fold → 赢家 LOO → 条件开启 Reserve**。

<a id="overview-round2" name="overview-round2"></a>

## 6. Round 2 第一阶段：按 family 做完整五折

第一阶段只运行 Active / Audit 对应的六组实验，共 30 fits。所有结果都与同 Fold FE10 配对比较：

| 排名 | 实验 | 研究方向 | Mean Δ vs FE10 | 正向 Fold | Δ Std | 当前判断 |
|---:|---|---|---:|---:|---:|---|
| 1 | `R_GROUP` | FE20 + FE22 + FE25 residual | **`+0.000079`** | **5/5** | `0.000062` | 进入 LOO |
| 2 | `O_GROUP` | FE28 + FE29 ratio Audit | `+0.000048` | 4/5 | `0.000073` | 边缘正信号，暂不扩展 |
| 3 | `FE26` | leisure − work/study | `+0.000004` | 2/5 | `0.000071` | 接近零，关闭 |
| 4 | `CAT2_GROUP` | 二阶类别组合 | `-0.000028` | 2/5 | `0.000048` | 无增量，Reserve 冻结 |
| 5 | `FE27` | daily/weekend screen mean | `-0.000084` | 0/5 | `0.000046` | 方向一致为负，关闭 |
| 6 | `CAT_NUM_GROUP` | gated cat × num | `-0.000279` | 0/5 | `0.000062` | 明显负向，Reserve 冻结 |

`R_GROUP` 是唯一 5/5 Fold 正向的方向，但它的平均增量仍不大，单 Fold 最大增量也处于 NULL 的极值范围内。因此当前结论不是“FE20、FE22、FE25 都有效”，而是：

> residual family 展现了最一致的正方向，值得用 LOO 查明究竟是哪一个成员贡献增量。

`O_GROUP` 虽为 4/5 正向，但均值与 NULL-A 的平均绝对漂移接近，暂时不足以恢复整个 ratio family。其余四组没有进入第二阶段的证据。

<a id="overview-next" name="overview-next"></a>

## 7. 当前决策与后续路线

当前实验树已经明显收窄：

```text
R_GROUP 5/5 正向
→ 分别移除 FE20、FE22、FE25 做 LOO
→ 找到真正贡献成员
→ 只有证据足够时才考虑 FE21、FE23、FE24
→ 与其他已确认赢家组成 Round 2 Best Feature Set
→ 使用 random_state=2024 再跑完整 5-Fold
→ 通过后才考虑 Kaggle submission
```

| 支线 | 当前动作 | 不做什么 |
|---|---|---|
| R_GROUP | 进入 LOO | 不直接把三个成员全部收入最终集合 |
| O_GROUP | 保留为边缘 Audit 记录 | 不自动扩展 ratio family |
| FE26 / FE27 | 关闭 | 不继续衍生相似线性轴或 screen mean |
| CAT_NUM_GROUP | 冻结相关 Reserve | 不逐个从失败 group 中追噪声 |
| CAT2_GROUP | FE46 保持冻结 | 不启动三阶类别组合 |
| MCC01 | 等 FE 收束后独立测试 | 不把模型参数变化冒充特征工程 |

### 承前启后的小总结

Round 2 第一阶段的价值不只是找到一个小幅正向的 R_GROUP，更重要的是用 NULL 和完整五折排除了大量看似合理、实际没有增量的方向。下一步不再扩大搜索面，而是把计算预算集中在 R_GROUP 的归因和稳健性确认上。

<a id="overview-files" name="overview-files"></a>

## 8. 文件分工与关键数字

### 文件分工

| 文件 | 负责回答的问题 |
|---|---|
| `00_project_overview.ipynb` | 项目为什么按当前顺序推进，证据如何收敛 |
| `01_eda.ipynb` | 数据是什么，有哪些质量问题和建模线索 |
| `02_baseline.ipynb` | 第一条训练与提交流程能否跑通 |
| `03_cv_experiments.ipynb` | 本地验证是否可靠，CatBoost 需要训练多久 |
| `04_feature_engineering.ipynb` | Round 1 特征假设、筛选证据与 FE10 决策 |
| `05_feature_engineering_2.ipynb` | Round 2 的 NULL 校准、FE20–FE46、6 个第一阶段实验与 R_GROUP LOO |
| `experiments.md` | 已确认实验的简明事实记录 |
| `src/` | 特征定义、训练协议和结果汇总的执行来源 |
| `results/` | 每个 Fold 的原始证据与阶段汇总 |

### 关键数字速查

| 数字 | 含义 |
|---:|---|
| `0.964007 ± 0.000473` | 无特征工程 CatBoost 正式 5-Fold BASE |
| `0.964629 ± 0.000523` | FE10 正式 5-Fold Reference |
| `+0.000621` | FE10 相对 BASE 的平均增量，5/5 Fold 为正 |
| `0.96606` | FE10 Public LB |
| `0.000051–0.000122` | 三组 NULL 的 mean absolute delta 范围 |
| `0.000182` | 当前 NULL 观察到的最大单 Fold absolute delta |
| `+0.000079` | R_GROUP 相对 FE10 的平均增量，5/5 Fold 为正 |

当前最值得保留的实验原则是：**先固定 Reference，再测噪声；先验证 family，再拆解成员；只有经过完整证据链的增量，才进入最终特征集合。**